# RGB v2 Feature Extraction — CIFAR-10

Load the trained RGB v2 CNN, recreate the original split, and extract `svm_features` for train, validation, and official test. This notebook does not fit models or evaluate accuracy.


## 1. Import dan Reproducibility

Random state harus sama dengan RGB Training supaya train/validation split identik.

In [24]:
import os
import random
import numpy as np

SEED = 30092026

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf

# Lebih aman untuk GPU dengan VRAM terbatas.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

tf.random.set_seed(SEED)

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Model, load_model
from sklearn.model_selection import train_test_split

print("SEED:", SEED)
print("GPU :", gpus)

SEED: 30092026
GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Konfigurasi File

CNN yang digunakan adalah model hasil dari notebook RGB Training.

In [25]:
CNN_MODEL_PATH = "cifar10_custom_cnn_v2.keras"
FEATURE_OUTPUT_PATH = "rgb_features_v2.npz"
FEATURE_LAYER = "svm_features"

if not os.path.exists(CNN_MODEL_PATH):
    raise FileNotFoundError(
        f"{CNN_MODEL_PATH} tidak ditemukan. "
        "Jalankan RGB Training terlebih dahulu dan letakkan file model "
        "pada folder yang sama dengan notebook ini."
    )

print("CNN model     :", CNN_MODEL_PATH)
print("Feature output:", FEATURE_OUTPUT_PATH)
print("Feature layer :", FEATURE_LAYER)

CNN model     : cifar10_custom_cnn_v2.keras
Feature output: rgb_features_v2.npz
Feature layer : svm_features


## 3. Load CIFAR-10 dan Recreate Split

RGB Training menggunakan:
- 50.000 official training images,
- 80% untuk training = 40.000,
- 20% untuk validation = 10.000,
- `random_state = 30092026`,
- `stratify = y_train_full`.

Official test set tetap 10.000 images.

Split dibuat kembali dengan parameter yang sama agar feature vector memiliki urutan sample yang konsisten dengan RGB Training.

In [26]:
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

# Simpan indeks agar alignment sample dapat diverifikasi pada
# RGB + AVG + NTSC Feature Engineering.
all_indices = np.arange(len(X_train_full))

train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full
)

X_train = X_train_full[train_indices].astype("float32") / 255.0
X_val = X_train_full[val_indices].astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

y_train = y_train_full[train_indices]
y_val = y_train_full[val_indices]

print("Training   :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)

assert X_train.shape == (40000, 32, 32, 3)
assert X_val.shape == (10000, 32, 32, 3)
assert X_test.shape == (10000, 32, 32, 3)

Training   : (40000, 32, 32, 3) (40000,)
Validation : (10000, 32, 32, 3) (10000,)
Test       : (10000, 32, 32, 3) (10000,)


## 4. Load Trained RGB CNN

`compile=False` digunakan karena notebook ini hanya melakukan inference / feature extraction.

In [27]:
tf.keras.backend.clear_session()

cnn_model = load_model(
    CNN_MODEL_PATH,
    compile=False
)

if FEATURE_LAYER not in [layer.name for layer in cnn_model.layers]:
    raise ValueError(
        f"Layer '{FEATURE_LAYER}' tidak ditemukan pada CNN."
    )

print("CNN berhasil dimuat.")
print("Input shape        :", cnn_model.input_shape)
print(
    "Feature layer shape:",
    cnn_model.get_layer(FEATURE_LAYER).output.shape
)

CNN berhasil dimuat.
Input shape        : (None, 32, 32, 3)
Feature layer shape: (None, 512)


## 5. Buat CNN Feature Extractor

Output CNN normal adalah `cnn_softmax`.

Untuk feature engineering dan SVM, output diambil dari layer `svm_features`, yaitu proyeksi Dense + BatchNormalization tanpa ReLU setelah `GlobalAveragePooling2D`.

```text
CIFAR-10 RGB image
        ↓
Trained RGB CNN
        ↓
svm_features
        ↓
512-dimensional feature vector
```

In [28]:
feature_extractor = Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer(
        FEATURE_LAYER
    ).output,
    name="rgb_cnn_feature_extractor"
)

feature_extractor.summary()

Model: "rgb_cnn_feature_extractor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 32, 32,    │      1,728 │ input_image[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 32, 32,    │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_relu           │ (None, 32, 32,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_conv1 │ (None, 32, 32,    │     36,864 │ stem_relu[0][0]   │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_bn1   │ (None, 32, 32,    │        256 │ stage1_block1_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_relu1 │ (None, 32, 32,    │          0 │ stage1_block1_bn… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_conv2 │ (None, 32, 32,    │     36,864 │ stage1_block1_re… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_bn2   │ (None, 32, 32,    │        256 │ stage1_block1_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_add   │ (None, 32, 32,    │          0 │ stage1_block1_bn… │
│ (Add)               │ 64)               │            │ stem_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_out   │ (None, 32, 32,    │          0 │ stage1_block1_ad… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_conv1 │ (None, 32, 32,    │     36,864 │ stage1_block1_ou… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_bn1   │ (None, 32, 32,    │        256 │ stage1_block2_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_relu1 │ (None, 32, 32,    │          0 │ stage1_block2_bn… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_conv2 │ (None, 32, 32,    │     36,864 │ stage1_block2_re… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_bn2   │ (None, 32, 32,    │        256 │ stage1_block2_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_add   │ (None, 32, 32,    │          0 │ stage1_block2_bn

 Total params: 11,442,624 (43.65 MB)

 Trainable params: 11,432,000 (43.61 MB)

 Non-trainable params: 10,624 (41.50 KB)

## 6. Extract RGB Features

Feature extraction dilakukan untuk:
- training set,
- validation set,
- official test set.

Hasilnya akan menjadi matrix dengan 512 feature untuk setiap image.

In [29]:
BATCH_SIZE = 128

X_train_features = feature_extractor.predict(
    X_train,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_val_features = feature_extractor.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_test_features = feature_extractor.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_train_features = X_train_features.astype(np.float32)
X_val_features = X_val_features.astype(np.float32)
X_test_features = X_test_features.astype(np.float32)

print()
print("RGB train features :", X_train_features.shape)
print("RGB val features   :", X_val_features.shape)
print("RGB test features  :", X_test_features.shape)

313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step

RGB train features : (40000, 512)
RGB val features   : (10000, 512)
RGB test features  : (10000, 512)


## 7. Validasi Feature Matrix

Untuk model RGB saat ini, layer `svm_features` menghasilkan 512-dimensional feature vector.

In [30]:
FEATURE_DIM = int(
    cnn_model.get_layer(FEATURE_LAYER).output.shape[-1]
)

assert X_train_features.shape == (
    len(X_train),
    FEATURE_DIM
)

assert X_val_features.shape == (
    len(X_val),
    FEATURE_DIM
)

assert X_test_features.shape == (
    len(X_test),
    FEATURE_DIM
)

assert len(y_train) == len(X_train_features)
assert len(y_val) == len(X_val_features)
assert len(y_test) == len(X_test_features)

assert np.isfinite(X_train_features).all()
assert np.isfinite(X_val_features).all()
assert np.isfinite(X_test_features).all()

print("Feature dimension :", FEATURE_DIM)
print("Semua shape valid.")
print("Tidak ditemukan NaN/Inf pada extracted features.")

Feature dimension : 512
Semua shape valid.
Tidak ditemukan NaN/Inf pada extracted features.


## 8. Simpan ke `rgb_features_v2.npz`

Satu archive menyimpan semua data yang diperlukan untuk Feature Engineering nanti:

- `X_train_features`
- `X_val_features`
- `X_test_features`
- `y_train`
- `y_val`
- `y_test`
- `train_indices`
- `val_indices`
- metadata sederhana (`seed`, `feature_dim`, `feature_layer`)

`train_indices` dan `val_indices` disimpan supaya alignment RGB, AVG, dan NTSC dapat diverifikasi sebelum feature concatenation.

In [31]:
np.savez_compressed(
    FEATURE_OUTPUT_PATH,
    X_train_features=X_train_features,
    X_val_features=X_val_features,
    X_test_features=X_test_features,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    train_indices=train_indices,
    val_indices=val_indices,
    seed=np.array(SEED, dtype=np.int64),
    feature_dim=np.array(FEATURE_DIM, dtype=np.int64),
    feature_layer=np.array(FEATURE_LAYER),
    representation=np.array("RGB")
)

print("Feature archive berhasil disimpan:")
print(FEATURE_OUTPUT_PATH)

Feature archive berhasil disimpan:
rgb_features_v2.npz


## 9. Verifikasi File Hasil

File dibuka kembali untuk memastikan archive dapat dibaca dan seluruh array tersimpan dengan benar.

In [32]:
with np.load(FEATURE_OUTPUT_PATH) as rgb:
    print("Keys:")
    for key in rgb.files:
        print(
            f"- {key}:",
            rgb[key].shape
            if rgb[key].ndim > 0
            else rgb[key].item()
        )

    assert np.array_equal(
        rgb["train_indices"],
        train_indices
    )
    assert np.array_equal(
        rgb["val_indices"],
        val_indices
    )
    assert np.array_equal(
        rgb["y_train"],
        y_train
    )
    assert np.array_equal(
        rgb["y_val"],
        y_val
    )
    assert np.array_equal(
        rgb["y_test"],
        y_test
    )

print()
print("Verifikasi selesai: rgb_features_v2.npz valid.")

Keys:
- X_train_features: (40000, 512)
- X_val_features: (10000, 512)
- X_test_features: (10000, 512)
- y_train: (40000,)
- y_val: (10000,)
- y_test: (10000,)
- train_indices: (40000,)
- val_indices: (10000,)
- seed: 30092026
- feature_dim: 512
- feature_layer: svm_features
- representation: RGB

Verifikasi selesai: rgb_features_v2.npz valid.


# RGB v2 extraction complete

Output: `rgb_features_v2.npz`. Run the combination notebook after this file is created.
